In [ ]:
!pip install -i https://test.pypi.org/simple/ lowresource-llm-evaluation==0.2.1
!pip install --upgrade sacrebleu
!pip install --upgrade transformers
!pip install python-Levenshtein
!pip install bitsandbytes accelerate

In [ ]:
import os
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from lowresource_llm_evaluation import benchmark
from lowresource_llm_evaluation.interferenciaLinguistica import loadLexicon
import pandas as pd
import numpy as np
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from google.colab import drive
import time
import json

drive.mount('/content/drive')
# load_dotenv("secrets.env")
# login(token=os.getenv("HF_TOKEN"))
base = "drive/MyDrive/EvaluacionTFG/"

In [ ]:
def load_gallego():
    with open(base + "EvalDatasets/Raw/idioms_train_es.txt", "r", encoding="utf-8") as fEsp:
        esp = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_train_gl.txt", "r", encoding="utf-8") as fGl:
        gl = fGl.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_es.txt", "r", encoding="utf-8") as fEsp:
        espTest = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_gl.txt", "r", encoding="utf-8") as fGl:
        glTest = fGl.readlines()
    return pd.DataFrame(np.array((esp + espTest, gl + glTest)).T, columns=["es","gl"])

def evaluate_benchmark(model_name, idioma, N= 20, debug=False):
    
    # 1. Define the 4-bit quantization configuration
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # 3. Load the pre-trained language model with quantization
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        trust_remote_code=True,
        tie_word_embeddings=False # Added to silence the warning about tied weights
    )

    print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
    print(f"Model loaded with 4-bit quantization: {model.__class__.__name__}")
    print(f"Model device: {model.device}")
    codigos = {"aranes": "aran" , 
               "asturiano": "ast", 
               "gallego": "gl"}
    df_textos = {"aranes": pd.read_parquet("hf://datasets/projecte-aina/ES-OC_Parallel_Corpus/es-arn_corpus.parquet").sample(N) , 
            "asturiano": pd.read_parquet("hf://datasets/projecte-aina/ES-AST_Parallel_Corpus/es-ast_corpus.parquet").sample(N), 
            "gallego": load_gallego() }
    return benchmark(model, tokenizer, 
            df_textos = df_textos[idioma],
            lang_eval= codigos[idioma],
            df_huecos=  pd.read_csv(base + f"EvalDatasets/Huecos/{idioma}.csv").sample(N),
            df_anotado = pd.read_csv(base + f"EvalDatasets/Anotado/{idioma}.csv").sample(N),
            lexicon_target = loadLexicon(base + f"lexicons/{codigos[idioma]}.txt"),
            lexicons_comparison = {"es": loadLexicon(base + f"lexicons/es.txt"), "fr": loadLexicon(base + f"lexicons/fr.txt")},
            roundtrip_langs= ["es"],
            debug=debug)

# Aranés

## Mistral 7B 

In [ ]:
idioma = "aranes"
modelo = "mistralai/Mistral-7B-Instruct-v0.3"
resultados = evaluate_benchmark(modelo, idioma, 10)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split("/")[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

In [ ]:
idioma = "aranes"
modelo = "BSC-LT/salamandra-7b-instruct"
resultados = evaluate_benchmark(modelo, idioma)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split("/")[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)

In [ ]:
idioma = "aranes"
modelo = "google/gemma-7b"
resultados = evaluate_benchmark(modelo, idioma)
date = time.localtime(time.time())
filename = f"resultados_{idioma}_{modelo.split("/")[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"
with open(base + filename, "w") as f:
    json.dump(resultados, f)